# Add Menu Item — Paste Recipe → menus.json

Paste any recipe below (Malay, English, or just ingredients + dish name). GPT-4o will extract the structured data and append it to `menus.json`.

**Steps:**
1. Cell 1 — setup
2. **Cell 2 — PASTE YOUR RECIPE HERE**
3. Cell 3 — GPT extracts structure (preview)
4. Cell 4 — writes to menus.json (confirm first!)


In [ ]:
# Cell 1 — Setup
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'openai', 'python-dotenv', '-q'])

import json, os
from pathlib import Path
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv('../backend/.env')

client = AzureOpenAI(
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT', 'https://kenduriluhh-ai-service.openai.azure.com/'),
    api_version=os.getenv('AZURE_OPENAI_API_VERSION', '2025-01-01-preview'),
)
DEPLOYMENT = os.getenv('AZURE_OPENAI_DEPLOYMENT', 'gpt-4o-kenduri')
KB_PATH = Path('../backend/app/knowledge_base/menus.json')

print('✅ Connected to Azure OpenAI')
print(f'   Deployment: {DEPLOYMENT}')
print(f'   menus.json: {KB_PATH.resolve()}')

In [ ]:
# Cell 2 — PASTE YOUR RECIPE HERE
# Can be: full recipe text, just ingredients, dish name + brief description
# In Malay, English, or mixed — GPT will handle it

RECIPE_INPUT = """
PASTE YOUR RECIPE HERE

Example:
Ayam Percik Kelantan
Chicken marinated in coconut milk, lemongrass, galangal, turmeric and dried chilli paste,
then grilled over charcoal and basted with percik sauce.
Serves 5 people per chicken (1.2kg).
Prep: 2 hours marinating + 45 min grill.
Suitable for: kenduri, wedding, rewang.
Main ingredients: ayam, santan, serai, lengkuas, kunyit, cili kering.
"""

# For Rewang mode - what household quantity makes sense?
REWANG_QUANTITY_NOTE = """e.g. 1 ekor ayam (1.2kg) untuk 5 orang"""

print('Recipe input set. Run Cell 3 to extract structure.')
print(f'Preview: {RECIPE_INPUT[:100].strip()}...')

In [ ]:
# Cell 3 — GPT extracts structured data (PREVIEW — nothing written yet)

with open(KB_PATH, encoding='utf-8') as f:
    existing_menus = json.load(f)['menus']

existing_ids = [m['id'] for m in existing_menus]
existing_names = [m['name'] for m in existing_menus]

SCHEMA_EXAMPLE = {
    "id": "ayam-percik-kelantan",
    "name": "Ayam Percik Kelantan",
    "name_en": "Kelantan Grilled Spiced Chicken",
    "category": "main_dish",
    "suitable_for": ["kenduri", "wedding", "rewang"],
    "halal_status": "halal",
    "min_pax": 10,
    "portion_per_pax_gram": 200,
    "raw_protein_per_pax_gram": 240,
    "prep_time_hours": 2.75,
    "cook_time_hours": 0.75,
    "rewang_measurement": "1 ekor ayam (1.2kg) untuk 5 orang",
    "katering_unit_cost_myr": 7.50,
    "main_ingredients": ["ayam_standard", "santan_segar", "serai", "lengkuas", "cili_kering"],
    "notes": "Marinate minimum 2 hours. Best grilled over charcoal for authentic flavour."
}

prompt = f"""You are a Malaysian culinary expert and data engineer.
Extract structured menu data from this recipe and return ONLY valid JSON — no markdown, no explanation.

EXISTING DISH IDs (do not duplicate): {existing_ids}
EXISTING DISH NAMES: {existing_names}

REQUIRED JSON SCHEMA (match exactly):
{json.dumps(SCHEMA_EXAMPLE, indent=2)}

RULES:
- id: lowercase-hyphenated, unique, descriptive (e.g. ayam-percik-kelantan)
- category: one of [carbohydrate, main_dish, side_dish, dessert, beverage, condiment]
- suitable_for: subset of [wedding, aqiqah, birthday, corporate, kenduri, rewang]
- halal_status: "halal" unless recipe contains haram ingredients
- katering_unit_cost_myr: realistic RM per person based on ingredient costs in Malaysia 2025
- rewang_measurement: use METRIC ONLY (kg, g, L, ml) — e.g. "1 ekor ayam (1.2kg) untuk 5 orang"
- main_ingredients: use snake_case IDs matching ingredients.json where possible
- If any field is uncertain, make a reasonable professional estimate

RECIPE:
{RECIPE_INPUT}

REWANG QUANTITY NOTE: {REWANG_QUANTITY_NOTE}
"""

response = client.chat.completions.create(
    model=DEPLOYMENT,
    messages=[{'role': 'user', 'content': prompt}],
    temperature=0.2,
    max_tokens=800,
)

raw_output = response.choices[0].message.content.strip()

# Clean up if GPT wrapped in markdown
if raw_output.startswith('```'):
    raw_output = raw_output.split('```')[1]
    if raw_output.startswith('json'):
        raw_output = raw_output[4:]

try:
    extracted = json.loads(raw_output)
    print('✅ Extracted successfully:\n')
    print(json.dumps(extracted, ensure_ascii=False, indent=2))
    print(f'\nDish ID: {extracted["id"]}')
    print(f'Katering cost: RM{extracted["katering_unit_cost_myr"]}/pax')
    print(f'Category: {extracted["category"]}')
    print('\nRun Cell 4 to append to menus.json')
except json.JSONDecodeError as e:
    print(f'❌ JSON parse error: {e}')
    print('Raw GPT output:')
    print(raw_output)

In [ ]:
# Cell 4 — WRITE to menus.json
# Only run after reviewing Cell 3 output!

# Safety check
if 'extracted' not in dir():
    raise RuntimeError('Run Cell 3 first!')

if extracted['id'] in existing_ids:
    raise ValueError(f'Dish ID "{extracted["id"]}" already exists! Change it in Cell 3 output or update manually.')

# Append and write
with open(KB_PATH, encoding='utf-8') as f:
    menus_data = json.load(f)

menus_data['menus'].append(extracted)

with open(KB_PATH, 'w', encoding='utf-8') as f:
    json.dump(menus_data, f, ensure_ascii=False, indent=2)

print(f'✅ Added "{extracted["name"]}" to menus.json')
print(f'   Total menus now: {len(menus_data["menus"])}')
print(f'   ID: {extracted["id"]}')
print(f'\nThe RAG service will pick it up automatically on next server restart (lru_cache resets).')

In [ ]:
# Cell 5 (OPTIONAL) — Add multiple recipes in batch
# Set BATCH_RECIPES as a list of recipe strings, runs Cell 3+4 logic for each

BATCH_RECIPES = [
    # Add your recipes here, one per list item
    # "Recipe 1 text...",
    # "Recipe 2 text...",
]

if not BATCH_RECIPES:
    print('No batch recipes defined. Add to BATCH_RECIPES list above.')
else:
    with open(KB_PATH, encoding='utf-8') as f:
        menus_data = json.load(f)
    
    added = []
    for i, recipe_text in enumerate(BATCH_RECIPES):
        print(f'Processing recipe {i+1}/{len(BATCH_RECIPES)}...')
        
        batch_prompt = prompt.replace(RECIPE_INPUT, recipe_text)
        batch_response = client.chat.completions.create(
            model=DEPLOYMENT,
            messages=[{'role': 'user', 'content': batch_prompt}],
            temperature=0.2, max_tokens=800,
        )
        raw = batch_response.choices[0].message.content.strip()
        if raw.startswith('```'):
            raw = raw.split('```')[1]
            if raw.startswith('json'): raw = raw[4:]
        try:
            item = json.loads(raw)
            existing_batch_ids = [m['id'] for m in menus_data['menus']]
            if item['id'] not in existing_batch_ids:
                menus_data['menus'].append(item)
                added.append(item['name'])
                print(f'  ✅ {item["name"]} ({item["id"]})')
            else:
                print(f'  ⚠️  {item["id"]} already exists, skipped')
        except Exception as e:
            print(f'  ❌ Failed to parse recipe {i+1}: {e}')
    
    with open(KB_PATH, 'w', encoding='utf-8') as f:
        json.dump(menus_data, f, ensure_ascii=False, indent=2)
    
    print(f'\n✅ Batch complete. Added {len(added)} recipes: {added}')